# Lab 03 — Noise Modeling & Spatial Denoising

**Course:** Image Analysis and Processing — École des Mines de Saint-Étienne  
**Topics covered:**
1. Synthetic generation of canonical noise distributions: Uniform, Gaussian, Salt & Pepper, and Exponential
2. Spatial denoising filters (Mean, Median, Minimum, Maximum) evaluated on corrupted images
3. Quantitative evaluation via Peak Signal-to-Noise Ratio (PSNR)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, img_as_float
from skimage.metrics import peak_signal_noise_ratio as psnr
from scipy.ndimage import (
    minimum_filter,
    maximum_filter,
    uniform_filter,
    median_filter,
)

rng = np.random.default_rng(42)

## 1. Pure Noise Distributions

We synthesize four common noise models and verify their probability density functions via normalized histograms.

In [ ]:
def plot_histogram(ax, values, title, bins=50, color="steelblue"):
    ax.hist(values.ravel(), bins=bins, color=color, alpha=0.75, density=True)
    ax.set_title(title)
    ax.set_xlabel("Value")
    ax.set_ylabel("Density")
    ax.grid(axis="y", alpha=0.4)


shape = (256, 256)
U = rng.random(shape)

# 1. Uniform noise on [0.2, 0.8]
noise_uniform = 0.2 + (0.8 - 0.2) * U

# 2. Gaussian noise N(0.5, 0.1^2)
noise_gaussian = 0.5 + 0.1 * rng.standard_normal(shape)

# 3. Salt & Pepper impulse noise
noise_sp = np.zeros_like(U)
noise_sp[(U > 0.1) & (U <= 0.9)] = 0.5
noise_sp[U > 0.9] = 1.0

# 4. Exponential noise (lambda = 5.0)
noise_exponential = -(1.0 / 5.0) * np.log(1.0 - U + 1e-10)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
plot_histogram(axes[0], noise_uniform, "Uniform Noise")
plot_histogram(axes[1], noise_gaussian, "Gaussian Noise")
plot_histogram(axes[2], noise_sp, "Salt & Pepper Noise", bins=10)
plot_histogram(axes[3], noise_exponential, "Exponential Noise")
fig.suptitle("Synthetic Noise Distributions", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 2. Spatial Denoising Comparison

We corrupt the standard `camera` benchmark image with:
- **Additive Gaussian noise** ($\sigma = 0.08$)
- **Salt & Pepper impulse noise** (10% corrupted pixels)

and compare 3x3 Mean, Median, Minimum, and Maximum spatial filters using PSNR.

In [ ]:
image = img_as_float(data.camera())

# Gaussian corrupted image
img_gauss = np.clip(image + 0.08 * rng.standard_normal(image.shape), 0.0, 1.0)

# Salt & Pepper corrupted image (5% salt, 5% pepper)
u_sp = rng.random(image.shape)
img_sp = image.copy()
img_sp[u_sp < 0.05] = 0.0
img_sp[u_sp > 0.95] = 1.0


def evaluate_filters(noisy_img, clean_img, title):
    filtered = {
        "Noisy": noisy_img,
        "Mean (3x3)": uniform_filter(noisy_img, size=3),
        "Median (3x3)": median_filter(noisy_img, size=3),
        "Min (3x3)": minimum_filter(noisy_img, size=3),
        "Max (3x3)": maximum_filter(noisy_img, size=3),
    }
    fig, axes = plt.subplots(1, len(filtered), figsize=(18, 4))
    for ax, (name, res) in zip(axes, filtered.items()):
        score = psnr(clean_img, res, data_range=1.0)
        ax.imshow(res, cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"{name}\nPSNR: {score:.2f} dB")
        ax.axis("off")
    fig.suptitle(title, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()


evaluate_filters(img_gauss, image, "Denoising Additive Gaussian Noise (sigma = 0.08)")
evaluate_filters(img_sp, image, "Denoising Salt & Pepper Noise (10% density)")